# S6E9: the generator remembers the original rows

*v3 (Sep 17): section 4 now holds income fixed (matched within \$100, own value out) and adds a within-cell contrast; v2 (Sep 16) reported the uncontrolled +0.064; v1 counted matches over twelve columns.*

The competition file was generated by a model trained on a 10,000-row original. This notebook shows that the generator did not only learn the original's *rules* — it kept a memory of the individual original rows — and that the exact-value part of that memory is what the feature engineering of this episode has been picking up without naming it. Four measurements, one short cell each (the hidden cell below computes the pieces they share), then what they are worth in AUC: less than you would think.

| # | claim | number |
|---|---|---|
| 1 | Train incomes are **copies** of the 10,000 original incomes | 97.9% of train rows carry an original income value; drawn independently it would be 8.4% |
| 2 | The buy rate of an exact income value is **not noise** around the income curve | variance 3.7× binomial against the income curve, 2.4× net of the other twelve columns → real per-value effects of about **±0.044** |
| 3 | Part of that effect is **the label of the original row** with that income | excess buy rate **+0.029** when the original row bought, **−0.009** when it did not; the label correlates with the residual more than any other attribute of that row |
| 4 | The memory is **row-level**, on top of the value | the generator copies the original's city, chargers and car type at 1.2–1.4× chance (the recipe cell only 1.08×); with the train row's cell and income fixed, an original buyer *in* the cell adds **+0.011** over one elsewhere, in 5 cells of 5 |
| 5 | What it is worth | OOF **0.94603** (raw columns 0.94180) from the encoding alone — prior 1, keys value / `//100` / `//1000` / nearest original, column sampling 0.3; the original's label adds 0.00003 (immaterial); income × recipe keys **cost** 0.0004 |


## 0. Setup — paths are found by name (Kaggle: competition data + [`itzzomkar/ev-adoption-behavior-and-range-anxiety`](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety) attached; or local). `FAST=True` subsamples the model section only. The two hidden cells below load the files and compute the headline panels; §1–4 show the pieces.

In [ ]:
import glob
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE, GREEN, GREY, RED = "#0072B2", "#E69F00", "#009E73", "#6B7785", "#D55E00"
SEED = 42
FAST = False   # True: the model section runs on 60k rows, 3 folds, lr 0.1 (minutes). Sections 1-4 always use the full file.
print("lightgbm", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)


def find(name):
    for root in ["/kaggle/input", "data", ".", ".."]:
        hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(name)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
sub = pd.read_csv(find("sample_submission.csv"))
orig = pd.read_csv(find("EV_Adoption_and_Range_Anxiety_Dataset.csv"))
assert train.shape == (668665, 15) and orig.shape == (10000, 15), "find() picked the wrong train.csv or original file"
train["y"] = (train.Will_Buy_EV == "Yes").astype(int)
orig["y"] = (orig.Will_Buy_EV == "Yes").astype(int)
print(f"train {train.shape}, test {test.shape}, original {orig.shape}; buy rate train {train.y.mean():.4f}, original {orig.y.mean():.4f}")

In [ ]:
# Everything the headline figure needs, computed once here and reused by the sections below.
RECIPE = ["Environmental_Concern_Level", "Subsidy_Available", "Range_Anxiety_Level"]
OTHER = ["Age", "Gender", "City_Type", "Number_of_Cars_Owned", "Current_Car_Type", "Daily_Commute_km",
         "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Home_Charging_Possible"]
orig_values = np.sort(orig.Annual_Income_USD.dropna().unique())
# original rows whose income is unique in the original file: the income identifies the row
oo = orig.dropna(subset=["Annual_Income_USD"]).drop_duplicates("Annual_Income_USD", keep=False).set_index("Annual_Income_USD")

# (a) per-value buy rate against a smooth income curve (300 quantile bins), the $30,000 floor excluded
g = train.groupby("Annual_Income_USD").y.agg(["sum", "size"])
g.columns = ["k", "n"]
g["rate"] = g.k / g.n
g["inc"] = g.index
g = g[g.inc > 30000]
g["bin"] = pd.qcut(g.inc.rank(method="first"), 300, labels=False)
smooth = g.groupby("bin").apply(lambda d: d.k.sum() / d.n.sum())
g["expected"] = g.bin.map(smooth)
g["resid"] = g.rate - g.expected
lab = g[(g.n >= 30) & g.inc.isin(oo.index)].copy()
lab["orig_label"] = oo.y.reindex(lab.inc).values
rng = np.random.default_rng(SEED)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(lab), len(lab))
    r = lab.resid.values[idx]
    l = lab.orig_label.values[idx]
    boot.append((r[l == 1].mean(), r[l == 0].mean()))
boot = np.array(boot)
m_bought, m_not = lab.resid[lab.orig_label == 1].mean(), lab.resid[lab.orig_label == 0].mean()
ci_bought, ci_not = np.percentile(boot[:, 0], [2.5, 97.5]), np.percentile(boot[:, 1], [2.5, 97.5])

# (b) row-level memory with the row's own cell AND income held fixed. Each train row is compared with its neighbours in
#     the same (concern, subsidy, anxiety) cell whose income lies within $100, its own income value left out. Then, within
#     each cell, rows whose original row sits IN that cell are contrasted with rows whose original row sits in ANOTHER cell.
m = train[train.Annual_Income_USD.isin(oo.index)].copy()
O = oo.reindex(m.Annual_Income_USD)
m["same_recipe"] = sum((m[c].values == O[c].values).astype(int) for c in RECIPE)
m["same_other"] = sum((m[c].values == O[c].values).astype(int) for c in OTHER)
m["orig_label"] = O.y.values
m["cell"] = m.Environmental_Concern_Level.astype(str) + m.Subsidy_Available + m.Range_Anxiety_Level
band = m.cell + "_" + (m.Annual_Income_USD // 100).astype(int).astype(str)
tot = m.groupby(band).y.agg(["sum", "size"])
own = m.groupby([band, m.Annual_Income_USD]).y.agg(["sum", "size"])
tot_s, tot_n = tot["sum"].reindex(band).values, tot["size"].reindex(band).values
pairs = list(zip(band, m.Annual_Income_USD))
own_s, own_n = own["sum"].reindex(pairs).values, own["size"].reindex(pairs).values
nb_n = tot_n - own_n
ok = nb_n >= 20
m["excess"] = np.where(ok, m.y.values - (tot_s - own_s) / np.maximum(nb_n, 1), np.nan)
mk = m[ok]
m["orig_in_cell"] = (m.same_recipe == 3)
# within-cell contrast, per cell, for the two original labels
contrast = []
for (lbl, cell), d in mk.groupby(["orig_label", "cell"]):
    a, b = d.excess[d.same_recipe == 3], d.excess[d.same_recipe < 3]
    if len(a) >= 100 and len(b) >= 100:
        contrast.append(dict(orig_label=lbl, cell=cell, in_cell=a.mean(), other_cell=b.mean(), n_in=len(a), n_other=len(b)))
contrast = pd.DataFrame(contrast)
contrast["diff"] = contrast.in_cell - contrast.other_cell


def wmean(d, col):
    return np.average(d[col], weights=d.n_in)


summary = {lbl: dict(in_cell=wmean(d, "in_cell"), other=wmean(d, "other_cell"), diff=wmean(d, "diff"),
                     cells=len(d), positive=int((d["diff"] > 0).sum())) for lbl, d in contrast.groupby("orig_label")}

fig, ax = plt.subplots(1, 2, figsize=(11, 4.0), gridspec_kw={"width_ratios": [1, 1.25]})
ax[0].bar([0, 1], [m_not, m_bought], color=[GREY, BLUE], width=0.6)
ax[0].errorbar([0, 1], [m_not, m_bought], yerr=[[m_not - ci_not[0], m_bought - ci_bought[0]], [ci_not[1] - m_not, ci_bought[1] - m_bought]],
               fmt="none", ecolor="black", capsize=5, lw=1.5)
ax[0].axhline(0, color="black", lw=0.8)
ax[0].set_xticks([0, 1])
ax[0].set_xticklabels([f"original row did NOT buy\n({(lab.orig_label == 0).sum():,} income values)",
                       f"original row BOUGHT\n({(lab.orig_label == 1).sum():,} income values)"])
ax[0].set_ylabel("buy rate of the exact income value\nminus the smooth income curve")
ax[0].set_title("The original row's label is in the train buy rate", loc="left", fontsize=11)
pos = {(1, "in"): 0, (1, "out"): 1, (0, "in"): 3, (0, "out"): 4}
for lbl, col in [(1, BLUE), (0, GREY)]:
    sm = summary[lbl]
    ax[1].bar([pos[(lbl, "in")], pos[(lbl, "out")]], [sm["in_cell"], sm["other"]], color=[col, "white"], edgecolor=col, lw=2, width=0.7)
    d = contrast[contrast.orig_label == lbl]
    ax[1].scatter(np.full(len(d), pos[(lbl, "in")]) + rng.uniform(-0.15, 0.15, len(d)), d.in_cell, color="black", s=14, zorder=3)
    ax[1].scatter(np.full(len(d), pos[(lbl, "out")]) + rng.uniform(-0.15, 0.15, len(d)), d.other_cell, color="black", s=14, zorder=3)
ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_ylim(top=ax[1].get_ylim()[1] * 1.3)
ax[1].set_xticks([0, 1, 3, 4])
ax[1].set_xticklabels(["in this\ncell", "in another\ncell", "in this\ncell", "in another\ncell"])
ax[1].text(0.5, ax[1].get_ylim()[1] * 0.97, f"original row BOUGHT\n+{summary[1]['diff']:.3f} in {summary[1]['positive']}/{summary[1]['cells']} cells",
           ha="center", va="top", color=BLUE, fontsize=10)
ax[1].text(3.5, ax[1].get_ylim()[1] * 0.97, f"original row did NOT buy\n{summary[0]['diff']:+.3f} in {summary[0]['positive']}/{summary[0]['cells']} cells",
           ha="center", va="top", color=GREY, fontsize=10)
ax[1].set_ylabel("excess buy rate vs neighbours\n(same cell, income within $100, own value out)")
ax[1].set_xlabel("where the original row with this income sits (dots: one per train cell)")
ax[1].set_title("...and the row leaves a trace on top of its value", loc="left", fontsize=11)
plt.tight_layout()
plt.show()
print(f"left:  {m_bought:+.4f} [{ci_bought[0]:+.4f}, {ci_bought[1]:+.4f}] vs {m_not:+.4f} [{ci_not[0]:+.4f}, {ci_not[1]:+.4f}]   "
      f"(original income values with >= 30 train rows, bootstrap 95% CI)")
print(f"right: original bought, in this cell {summary[1]['in_cell']:+.4f} vs in another cell {summary[1]['other']:+.4f} -> "
      f"{summary[1]['diff']:+.4f} more, positive in {summary[1]['positive']}/{summary[1]['cells']} cells | original did not buy: "
      f"{summary[0]['in_cell']:+.4f} vs {summary[0]['other']:+.4f} -> {summary[0]['diff']:+.4f}, {summary[0]['positive']}/{summary[0]['cells']} cells")

**Left:** every original income value with at least 30 train rows; its train buy rate minus the smooth income curve, split by what the *original* row with that income answered. **Right:** each train row against its neighbours in its own (concern, subsidy, anxiety) cell and within \$100 of income, its own income value left out; then, cell by cell, rows whose original row sits *in* that cell against rows whose original row sits in another.

**Credit.** The original's generator (seed 101, the purchase recipe) was reconstructed in Chris Deotte's [EDA](https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights). That 97.9% of train incomes fall on the original's finite value support was first noted, as a closing caution, in [Yusuke Hayashi's exact reconstruction of the original](https://www.kaggle.com/code/yhay81/exact-reconstruction-of-the-ev-source-dataset). That the score lives in the *resolution* of the income column is Georgy Mamarin's finding, in his [ledger](https://www.kaggle.com/code/georgymamarin/s6e9-what-the-board-paid-for-eleven-submissions); that the `% 1000` remainder of income predicts on its own (AUC 0.618) is [megayak's leak](https://www.kaggle.com/code/megayak/s6e9-the-1000-leak) — digit features were public before (cstdy, Dvorkin); the `_org_mean` features and the coarse income keys (Markus.JM's smooth keys) appear in [Naji's LightGBM](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637). What is new here is *why* those work, measured against controls. Nothing needs a GPU.

## 1. The incomes are copies

Original incomes are integers spread over \$30k–\$223k, so a fresh draw near any given income would land on an original value about one time in twelve. In train it is 98 times in 100, and each original value comes back about sixty times. The 2.1% of incomes that are not original values sit a couple of dollars from one — but the originals are dense enough that this distance alone cannot tell a jitter from a fresh draw, so the notebook does not lean on it. (Commute's 100% is what any generator on a 0.1 km grid gives: not evidence by itself.)

In [ ]:
rows = []
for c in ["Annual_Income_USD", "Daily_Commute_km"]:
    ov = set(orig[c].dropna())
    per_value = train[c].value_counts()
    rows.append(dict(column=c, distinct_train=train[c].nunique(), distinct_original=len(ov),
                     train_rows_on_an_original_value=f"{train[c].isin(ov).mean():.1%}",
                     test_rows_on_an_original_value=f"{test[c].isin(ov).mean():.1%}",
                     median_train_rows_per_original_value=int(per_value[per_value.index.isin(ov)].median())))
copies = pd.DataFrame(rows).set_index("column")
display(copies)

inv = train.Annual_Income_USD.value_counts()
inv = inv[~inv.index.isin(orig_values)]
j = np.clip(np.searchsorted(orig_values, inv.index.values), 1, len(orig_values) - 1)
dist = np.minimum(np.abs(orig_values[j] - inv.index.values), np.abs(orig_values[j - 1] - inv.index.values))
# the null: if a train income were an integer drawn near its own value but independently of the original file, how often
# would it land exactly on an original value? = local density of original values (per dollar) around each train row
tv = train.Annual_Income_USD.values
density = (np.searchsorted(orig_values, tv + 500, side="right") - np.searchsorted(orig_values, tv - 500, side="left")) / 1001
print(f"share of train rows whose income is an original value: {train.Annual_Income_USD.isin(orig_values).mean():.1%}   "
      f"expected if incomes were drawn independently (local density of original values): {density.mean():.1%}")
print(f"income values NOT in the original: {len(inv):,} distinct, {inv.sum():,} rows ({inv.sum() / len(train):.1%}), {inv.mean():.2f} rows per value; "
      f"median ${np.median(dist):.0f} from the nearest original value (the originals are dense: that distance alone does not separate a jitter from a fresh draw)")

## 2. The buy rate of an exact income value is not noise

Take every original income value with 30–120 train rows. Compute its buy rate, subtract what a smooth curve of income predicts at that point (300 quantile bins), and look at the **variance** of what is left. If the exact value carried no information beyond "how much money", that variance would be binomial sampling noise. It is 3.7 times larger (1.9× in standard deviation) — and 2.4× (±0.044) once the other twelve columns are controlled with a raw model's out-of-fold prediction, which is the part the exact value adds on top of everything else the model sees. Mamarin's §9 measures this wiggle net of binomial noise and shows it replicates across two halves of train; here it is re-measured with the \$30,000 floor excluded and net of the other columns. This is the number behind the tricks that raised the public board this month — `max_bin`, decimal digits, the `% 1000` remainder, frequency and target encoding — all ways of letting a model see the value rather than the bin (his reading; the measurement agrees).

In [ ]:
s = g[(g.n >= 30) & (g.n <= 120) & g.inc.isin(orig_values)]
binom = (s.expected * (1 - s.expected) / s.n).mean()
total = s.resid.var()
print(f"{len(s):,} original income values with 30-120 train rows (mean {s.n.mean():.0f} rows each)")
print(f"against the income curve:      variance of (buy rate - expectation) {total:.5f}   binomial alone {binom:.5f}   "
      f"ratio {total / binom:.2f} (x{np.sqrt(total / binom):.1f} in standard deviation)   real spread about +/-{np.sqrt(total - binom):.3f}")

# the same, net of everything else the row says: expectation = out-of-fold prediction of a raw 13-column LightGBM
# (255 bins, no encoding: a bin holds ~50 income values, so it cannot single one out) - the part the exact value adds
# on top of the other twelve columns. The curve version above also counts rows differing in those columns as "value effect".
from sklearn.model_selection import StratifiedKFold
raw = train[[c for c in test.columns if c != "id"]].copy()
for c in ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]:
    raw[c] = pd.Categorical(raw[c])
oof_raw = np.zeros(len(train))
p_raw = dict(objective="binary", metric="auc", learning_rate=0.1, num_leaves=31, min_data_in_leaf=100, feature_fraction=0.8,
             bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=255, seed=SEED, num_threads=os.cpu_count(), verbose=-1)
for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=42).split(raw, train.y):
    dtr = lgb.Dataset(raw.iloc[tr_i], train.y.values[tr_i])
    mdl = lgb.train(p_raw, dtr, 5000, valid_sets=[lgb.Dataset(raw.iloc[va_i], train.y.values[va_i], reference=dtr)],
                    callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_raw[va_i] = mdl.predict(raw.iloc[va_i], num_iteration=mdl.best_iteration)
g2 = train.assign(p=oof_raw, pq=oof_raw * (1 - oof_raw)).groupby("Annual_Income_USD").agg(k=("y", "sum"), n=("y", "size"), expected=("p", "mean"), pq=("pq", "mean"))
g2["resid"] = g2.k / g2.n - g2.expected
s2 = g2[(g2.n >= 30) & (g2.n <= 120) & g2.index.isin(orig_values) & (g2.index > 30000)]
binom2 = (s2.pq / s2.n).mean()   # per-row probabilities now vary within a value: the sampling variance is mean p(1-p)/n
total2 = s2.resid.var()
print(f"net of the raw model's prediction: variance {total2:.5f}   binomial alone {binom2:.5f}   ratio {total2 / binom2:.2f} "
      f"(x{np.sqrt(total2 / binom2):.1f} in SD)   real spread about +/-{np.sqrt(total2 - binom2):.3f}   (raw model OOF AUC {roc_auc_score(train.y, oof_raw):.4f})")
lab2 = s2[s2.index.isin(oo.index)]
ol2 = oo.y.reindex(lab2.index).values
print(f"section 3 net of the raw model: original bought {lab2.resid[ol2 == 1].mean():+.4f} vs did not buy {lab2.resid[ol2 == 0].mean():+.4f}")
null = rng.binomial(s.n.values, s.expected.values) / s.n.values - s.expected.values
fig, ax = plt.subplots(figsize=(7.5, 3.4))
bins = np.linspace(-0.3, 0.4, 71)
ax.hist(null, bins=bins, color=GREY, alpha=0.55, label="what binomial noise alone would give")
ax.hist(s.resid, bins=bins, color=BLUE, alpha=0.6, label="observed against the income curve")
ax.hist(s2.resid, bins=bins, color=ORANGE, alpha=0.5, label="observed, net of the raw model")
ax.set_xlabel("buy rate of the value  -  expectation")
ax.set_ylabel("income values")
ax.legend(frameon=False)
ax.set_title("The buy rate of an exact income value is not noise around a curve", loc="left", fontsize=11)
plt.tight_layout()
plt.show()

## 3. The original row's label is in that residual

Each of those income values belongs to exactly one original row (values shared by several original rows are left out). That row has a label. Split the per-value residual by it: the values whose original **row** said *Yes* have a higher buy rate in train than the curve predicts, the ones whose original said *No* a lower one, and the two bootstrap intervals do not come close. Among everything the original row could have passed on, the **label** (0.17) is the attribute the residual tracks best; the recipe's own purchase score comes next (0.12), so this is not merely the copied concern, subsidy and anxiety showing through. City (0.10) and the charging columns predict the original label barely at all (r ≈ 0.02, right-hand column): they show up here because the generator copies them from the original row — §4 measures that directly.

The label explains under three percent of the per-value variance. It is a witness to the memory rather than the memory itself: most of the ±0.044 is idiosyncratic to the generator and can only be read off the training rows, which is what a target encoding does.

In [ ]:
def recipe(df):
    return (1.2 * df.Annual_Income_USD / 1e5 + 0.6 * df.Environmental_Concern_Level + 2.0 * (df.Subsidy_Available == "Yes")
            - 1.0 * (df.Range_Anxiety_Level == "Medium") - 3.0 * (df.Range_Anxiety_Level == "High"))


oo["recipe_score"] = recipe(oo.reset_index()).values
d = g[(g.n >= 30) & g.inc.isin(oo.index)].copy()
O2 = oo.reindex(d.inc)
NAMES = {"y": "label (Will_Buy_EV)", "recipe_score": "purchase score of the recipe"}
cors = []
for c in ["y", "recipe_score"] + RECIPE + OTHER:
    v = O2[c]
    if not pd.api.types.is_numeric_dtype(v):
        # text column: score each level by its buy rate in the original file (Yes/No, Low/Medium/High, city types...)
        v = O2[c].map(orig.groupby(c).y.mean())
    v = v.astype(float).values
    ok2 = ~np.isnan(v)
    cors.append(dict(attribute=NAMES.get(c, c), r_with_the_per_value_residual=np.corrcoef(d.resid.values[ok2], v[ok2])[0, 1],
                     r_with_the_original_label=np.corrcoef(O2.y.values[ok2], v[ok2])[0, 1]))
cors = pd.DataFrame(cors).set_index("attribute").round(3)
display(cors)
print(f"{len(d):,} income values (>= 30 train rows, unique in the original)")
print(f"mean residual when the original row BOUGHT:  {m_bought:+.4f}  (95% CI {ci_bought[0]:+.4f} .. {ci_bought[1]:+.4f})")
print(f"mean residual when the original did NOT buy: {m_not:+.4f}  (95% CI {ci_not[0]:+.4f} .. {ci_not[1]:+.4f})")
X1 = np.c_[np.ones(len(d)), O2.y.values]
beta = np.linalg.lstsq(X1, d.resid.values, rcond=None)[0]
r2 = 1 - ((d.resid.values - X1 @ beta) ** 2).sum() / ((d.resid.values - d.resid.mean()) ** 2).sum()
print(f"share of the per-value variance the original label explains: {r2:.1%}  (a witness to the memory, not the memory itself)")

## 4. It is a memory of rows, on top of the value

First, a check that reads no label: the generator copies the original row's *other* columns along with its income — city 1.32× more often than chance, charging stations 1.3×, car type 1.17×; the recipe cell (concern, subsidy, anxiety) only 1.08×. So a train row is a partial copy of one original row, not just of its income.

Does that copy carry the label beyond the value? Compare each train row with its neighbours in the same (concern, subsidy, anxiety) cell whose income lies within \$100 — leaving the row's own income value out, so the per-value effect of §2 cannot leak in — and then, inside each cell, contrast rows whose original **buyer** sits *in* that cell with rows whose original buyer sits in another cell: +0.037 vs +0.026, **+0.011** more, in five buyer cells out of five (+0.004 for original non-buyers, 7 cells of 10). The rest of the +0.037 is the per-value effect of §3 scaled by the cell's base rate (58% in those cells). The row leaves a trace beyond its income value; the value carries the larger share — which is what §5 finds from the other side. Sharing the other nine columns does not move the buy rate (they carry no label), but they are where the copying shows.

The last table is the control's own control. Holding only the cell fixed gives +0.064, but that still lets income in: rows whose original buyer sits in their cell are about \$4,700 richer than their cell's mean. Controlling income at ~90 rows per group takes it to +0.043 — the honest part of the drop. Finer bins (27 and 9 rows) keep shrinking it for a mechanical reason: the row's own value sits inside the bin. Leaving the own value out gives +0.037; the within-cell contrast above is what separates the row from its value.

*(v1 of this section counted matches over all twelve columns and found a 0.01 → 0.75 gradient — the recipe columns in disguise: a row that matches a buyer's concern, subsidy and anxiety is buyer-like on its own. v2 held only the cell fixed, +0.064. The controls above are what survived.)*

In [ ]:
# (i) a check that reads no label: how often does the train row share each OTHER column with its original row,
#     against what independence would give (the train frequency of the original row's value)?
copy_rows = []
for c in OTHER + RECIPE:
    freq = train[c].value_counts(normalize=True)
    same = float((m[c].values == O[c].values).mean())
    indep = float(freq.reindex(O[c].values).fillna(0).mean())
    copy_rows.append(dict(column=c, same_as_original_row=same, expected_if_independent=indep, ratio=same / indep))
copy_tab = pd.DataFrame(copy_rows).set_index("column").round(3).sort_values("ratio", ascending=False)
print("how often the train row carries the same value as its original row (income-matched), vs independence:")
display(copy_tab)

# (ii) the within-cell contrast behind the right-hand panel, cell by cell
ct = contrast.copy()
ct["orig_label"] = ct.orig_label.map({1: "original bought", 0: "original did not buy"})
print("excess buy rate vs neighbours (same cell, income within $100, own value out): original row IN this cell vs in ANOTHER cell:")
display(ct.set_index(["orig_label", "cell"]).round(4))
for lbl, name in [(1, "original bought"), (0, "original did not buy")]:
    sm = summary[lbl]
    print(f"{name:22s}: in this cell {sm['in_cell']:+.4f}  in another cell {sm['other']:+.4f}  difference {sm['diff']:+.4f}  "
          f"({sm['positive']}/{sm['cells']} cells positive; weighted by rows)")

# (iii) the cell-only control at increasing income resolution, for the record: rows whose original buyer sits in their cell are
#       richer than their cell's mean, so "cell only" still lets income in; finer bins then shrink the number mechanically
#       because the row's own value sits inside the bin. The matched version (own value out) is the one the panel uses.
inc_gap = (m[(m.same_recipe == 3) & (m.orig_label == 1)].Annual_Income_USD - m[(m.same_recipe == 3) & (m.orig_label == 1)].cell.map(m.groupby("cell").Annual_Income_USD.mean())).mean()
print(f"rows whose original buyer sits in their cell are ${inc_gap:,.0f} above their cell's mean income")
sweep = []
for nb in [1, 50, 200, 1000]:
    key = m.cell if nb == 1 else m.cell + "_" + pd.qcut(m.Annual_Income_USD.rank(method="first"), nb, labels=False).astype(str)
    resid_sweep = m.y - m.y.groupby(key).transform("mean")
    tt = resid_sweep.groupby([m.orig_label, m.same_recipe]).mean().unstack(0)
    sweep.append(dict(income_control=("none (cell only)" if nb == 1 else f"{nb} income bins"), rows_per_group=int(m.groupby(key).size().median()),
                      same_cell_and_bought=round(tt.loc[3, 1], 4), same_cell_not_bought=round(tt.loc[3, 0], 4)))
sweep.append(dict(income_control="within $100, own value out", rows_per_group=int(np.median(nb_n[ok])),
                  same_cell_and_bought=round(float(mk[(mk.same_recipe == 3) & (mk.orig_label == 1)].excess.mean()), 4),
                  same_cell_not_bought=round(float(mk[(mk.same_recipe == 3) & (mk.orig_label == 0)].excess.mean()), 4)))
display(pd.DataFrame(sweep).set_index("income_control"))

## 5. What it is worth in AUC

Two kinds of columns follow from §2–§4, and only one of them earns its place:

* **the exact-value memory** as a cross-fitted target encoding of `Annual_Income_USD` and `Daily_Commute_km` — at the value, at coarser keys (`// 100`, `// 1000`, whole kilometres: Markus.JM's smooth keys via Naji) and at the *nearest original value* so the 2.1% invented incomes borrow their neighbour's memory. Prior weight 1 (0.5 gives 0.00005 back; heavy smoothing washes the effect out, see megayak's sweep). Rebuilt inside every fold with an inner split so no training row ever sees its own label.
* **the original row as a witness** — `original_memory()` below: its label, how many recipe columns the train row shares with it, the label *only when it sits in the same cell*, its purchase score, its anxiety, subsidy and city. These read no competition label, so they are computed once, outside the folds. `orig_y_Annual_Income_USD` is Naji's `_org_mean` under another name; Dvorkin, megayak (the recipe score from the original's coefficients, +0.00002) and Mamarin's §16 already measured that family at zero — what is new is the reading, not the number.

The model is one LightGBM on those columns — 32 leaves, depth 5, 10 rows per leaf, column sampling 0.3, `max_bin` 1024, learning rate 0.02, early stopping — the regularisation Naji's notebook settled on. No digits, no frequency encoding, no crosses, no blend.

**Ablation**, out-of-fold AUC on the full training set, same folds throughout, learning rate 0.1 so each row costs about a minute. The noise bar is what two columns of uniform random numbers do to the score through the identical pipeline — Mamarin's rule: a change inside the bar is not a change, and the bar is re-measured whenever the model's capacity changes. The bar is the cost of two junk columns (n = 2), not a confidence interval; seed replicas of the final additive model differ by 0.000002, so paired fold differences (5/5) are the second criterion, applied to every row alike. Measured with LightGBM 4.7.0; the Kaggle run prints its own number — expect the fifth decimal to move.

| model (LightGBM, 5 folds, seed 42, learning rate 0.1 unless stated) | OOF AUC | change |
|---|---|---|
| raw 13 columns, `max_bin` 255, 31 leaves, column sampling 0.8 | 0.94180 | — |
| + `max_bin` 4095 | 0.94358 | +0.00178 |
| raw + cross-fitted target encoding of exact income and commute, prior 20 (255 bins) | 0.94484 | +0.00304 vs raw |
| target encoding + `max_bin` 4095 | 0.94504 | +0.00020 |
| + `interaction_constraints` (each column alone: an additive model) | 0.94540 | +0.00036 |
| ↳ noise bar at this capacity (two runs, two random columns each) | 0.94534 / 0.94534 | −0.00006 / −0.00006 |
| + coarser encoding keys (`income // 100`, `// 1000`, whole km) | 0.94552 | +0.00012 |
| + encoding prior 20 → 10 / 5 / 2 / 1 | 0.94563 / 0.94570 / 0.94575 / 0.94577 | +0.00025 (20 → 1) |
| + nearest-original-value key, prior 1  **[additive reference]** | 0.94582 | +0.00005 |
| ↳ prior 0.5 instead of 1 | 0.94577 | −0.00005 |
| ↳ noise bar for the additive reference | 0.94580 / 0.94581 | −0.00002 / −0.00001 |
| ↳ + the original-row witness columns (all thirteen / only the label ones) | 0.94583 / 0.94585 | +0.00001 / +0.00003 |
| ↳ + encoding keyed by income × subsidy, × concern, × anxiety (the row memory as a feature) | 0.94538 | −0.00044 |
| ↳ + encoding keyed by income × full cell | 0.94525 | −0.00056 |
| ↳ + `_org_mean` of every column / the `% 1000` digit encodings / encoding of the 11 other columns / the encoding at three priors at once (Naji's triple encoding) | 0.94585 / 0.94583 / 0.94577 / 0.94583 | +0.00003 / +0.00001 / −0.00004 / +0.00001 |
| ↳ the same features without the constraint (31 leaves, column sampling 0.8) | 0.94576 | −0.00005 |
| same features, **no constraint**, 32 leaves / depth 5 / min 10 rows per leaf / column sampling 0.3 / L2 2, L1 0.071 / `max_bin` 1024  **[final]** | 0.94596 | +0.00014 vs the additive reference |
| ↳ noise bar for the final model | 0.94591 / 0.94588 | −0.00005 / −0.00008 |
| ↳ the same with the additive constraint back on | 0.94586 | −0.00010 |
| ↳ column sampling 0.3 → 0.8 / 0.5 / 0.2 | 0.94588 / 0.94592 / 0.94599 | −0.00008 / −0.00004 / +0.00003 |
| ↳ `max_bin` 1024 → 255 / 4095 | 0.94597 / 0.94595 | +0.00001 / −0.00001 |
| ↳ 63 leaves without the depth-5 cap / min 50 rows per leaf | 0.94587 / 0.94596 | −0.00009 / −0.00000 |
| ↳ + witness columns (label ones) / `_org_mean` / digit + other-column encodings | 0.94591 / 0.94595 / 0.94599 | −0.00005 / −0.00001 / +0.00003 |
| final model at learning rate 0.02 (the one this notebook trains) | 0.94603 | +0.00007 vs the same at 0.1 |


The reading, row by row.

* The bin cap and the target encoding are the same lever pulled two ways — resolution on the income column — and once the encoding is in, the cap is worth about a tenth of what it was (+0.00020 against +0.00178) and, in the final model, nothing at all (255, 1024 and 4095 bins tie).
* The prior weight of the encoding is the next lever, and it stops at 1. The coarser keys and the nearest-original key are small steps that clear the bar (megayak found extra keys negative in his pipeline; here they clear the bar by 0.00012).
* The **witness columns are worth nothing usable**: +0.00003 in the additive reference — a hair outside a very narrow bar, consistent across folds, a tenth of what the prior weight is worth — and −0.00005 in the final model, inside its wider one. That is §3 from the other side: the label explains a few percent of the per-value variance, and sixty training rows per value already estimate that variance directly. `_org_mean` columns, the `% 1000` digits and encodings of the other eleven columns land in the same place.
* The row memory of §4 **could not be made into a feature with this encoding**: keying it by income × subsidy, income × concern and income × anxiety costs 0.0004, by income × full cell 0.0006. Those keys have 3 to 30 rows each (income × subsidy 30, × anxiety 13, × concern 13, × full cell 3); the trace is real but the training set cannot resolve it per row, and the model overfits the attempt. senihbayankulu's cell-conditioned local rate and kNN prior on the original also added nothing.
* On column interactions I was wrong for a day, and the table keeps the record. With 31 leaves and column sampling 0.8, forbidding every interaction (`interaction_constraints`) is worth +0.00036 over the plain encoding — the additive model wins, as Mamarin's ledger found (he measured +0.00118 for the constraint without target encoding, and wrote the caveat himself: a positive number says the search for combinations costs more in fitting noise than it returns). With the full encoding the same constraint is worth only +0.00005. Regularise the free model instead — depth 5, ten rows per leaf, column sampling 0.3 — and it beats the additive reference by 0.00014, five folds out of five, outside its wider bar; put the constraint back on that model and it costs 0.00010. Which of the three regularisers does the work is not separated here (sampling 0.8 costs 0.00008, at the bar's edge; 0.5 and 0.2 tie). So combinations of columns are worth about a ten-thousandth, and the constraint was standing in for regularisation, not for a fact about the generator.

A witness confirms the mechanism; it does not improve the estimate.

In [ ]:
CATS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]


def original_memory(df, orig):
    """What the original file says about the row's exact income value. Reads no competition label, so it is computed
    once, outside the folds. NaN where the income is not an original value (2.1% of rows). Measured in the text: adds
    nothing on top of the target encoding, kept here so the ablation can be re-run."""
    out = pd.DataFrame(index=df.index)
    for k in ["Annual_Income_USD", "Daily_Commute_km"]:
        gk = orig.dropna(subset=[k]).groupby(k)["y"].agg(["mean", "size"])
        out[f"orig_y_{k}"] = df[k].map(gk["mean"]).astype("float32")
        out[f"orig_n_{k}"] = df[k].map(gk["size"]).fillna(0).astype("float32")
    Or = oo.reindex(df.Annual_Income_USD)
    same = sum((df[c].values == Or[c].values).astype("int8") for c in RECIPE).astype("float32")
    olab = Or["y"].values.astype("float32")
    out["orig_same_recipe"] = np.where(np.isnan(olab), np.nan, same)
    out["orig_label_same_cell"] = np.where(same == 3, olab, np.nan)
    out["orig_label_x_same"] = (olab - 0.5) * same
    out["orig_recipe_score"] = recipe(Or.reset_index()).values.astype("float32")
    out["orig_concern"] = Or["Environmental_Concern_Level"].values.astype("float32")
    out["orig_nmatch"] = sum((df[c].values == Or[c].values).astype("int8") for c in RECIPE + OTHER).astype("float32")
    out.loc[np.isnan(olab), "orig_nmatch"] = np.nan
    for c in ["Range_Anxiety_Level", "Subsidy_Available", "City_Type"]:
        out[f"orig_{c}"] = pd.Categorical(Or[c].values)
    return out


def target_encode(Xtr, ytr, others, cols, n_inner=5, alpha=1.0, seed=SEED):
    """Cross-fitted mean target per exact value. Training rows get an inner out-of-fold mean (their own label never
    enters their feature); every other frame gets the mean over the whole training fold."""
    from sklearn.model_selection import KFold
    prior = ytr.mean()
    res = [dict() for _ in range(1 + len(others))]
    for c in cols:
        enc = np.full(len(Xtr), np.nan, dtype="float32")
        for a, b in KFold(n_inner, shuffle=True, random_state=seed).split(Xtr):
            gk = pd.DataFrame({"k": Xtr[c].values[a], "y": ytr[a]}).groupby("k")["y"].agg(["sum", "size"])
            enc[b] = pd.Series(Xtr[c].values[b]).map((gk["sum"] + alpha * prior) / (gk["size"] + alpha)).fillna(prior).values
        res[0][f"te_{c}"] = enc
        gk = pd.DataFrame({"k": Xtr[c].values, "y": ytr}).groupby("k")["y"].agg(["sum", "size"])
        full = (gk["sum"] + alpha * prior) / (gk["size"] + alpha)
        for i, df in enumerate(others):
            res[i + 1][f"te_{c}"] = df[c].map(full).fillna(prior).values.astype("float32")
    return res


def nearest_original(values):
    """The original income value nearest to each income: identical for the 97.9% that are copies, a neighbour for the rest."""
    j = np.clip(np.searchsorted(orig_values, values), 1, len(orig_values) - 1)
    left, right = orig_values[j - 1], orig_values[j]
    return np.where(np.abs(values - left) <= np.abs(right - values), left, right).astype("float32")


FEATURES = [c for c in test.columns if c != "id"]
X = train[FEATURES].copy()
X_test = test[FEATURES].copy()
y = train.y.values
for c in CATS:
    cats = pd.Categorical(pd.concat([X[c], X_test[c]])).categories
    X[c] = pd.Categorical(X[c], categories=cats)
    X_test[c] = pd.Categorical(X_test[c], categories=cats)
USE_ORIGINAL_WITNESS = False   # the ablation in the text: these columns add nothing on top of the encoding. Flip to re-measure.
if USE_ORIGINAL_WITNESS:
    X = pd.concat([X, original_memory(train, orig)], axis=1)
    X_test = pd.concat([X_test, original_memory(test, orig)], axis=1)
    for c in ["orig_Range_Anxiety_Level", "orig_Subsidy_Available", "orig_City_Type"]:
        cats = pd.Categorical(pd.concat([X[c].astype(object), X_test[c].astype(object)])).categories
        X[c] = pd.Categorical(X[c].astype(object), categories=cats)
        X_test[c] = pd.Categorical(X_test[c].astype(object), categories=cats)
for df in (X, X_test):
    # coarser keys for the encoding, and the nearest original value so the 2.1% invented incomes borrow their neighbour's memory
    df["income_100"] = (df.Annual_Income_USD // 100).astype("float32")
    df["income_1000"] = (df.Annual_Income_USD // 1000).astype("float32")
    df["commute_km"] = np.floor(df.Daily_Commute_km).astype("float32")
    df["income_nearest_original"] = nearest_original(df.Annual_Income_USD.values)
TE_COLS = ["Annual_Income_USD", "Daily_Commute_km", "income_100", "income_1000", "commute_km", "income_nearest_original"]
TE_ALPHA = 1.0
print(f"{X.shape[1]} columns before target encoding, + {len(TE_COLS)} encoded inside the folds (prior weight {TE_ALPHA:g})")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


def run_cv(X, y, X_test, params, n_folds=5, te_cols=TE_COLS, rounds=50000, log=True):
    """StratifiedKFold(5, shuffle=True, random_state=42) over train.csv in file order: the split most OOF libraries in this
    episode share, so the out-of-fold file written at the end stacks row-for-row with theirs."""
    oof = np.zeros(len(y))
    pred = np.zeros(len(X_test))
    fold_id = np.zeros(len(y), int)
    best = []
    for k, (tr, va) in enumerate(StratifiedKFold(n_folds, shuffle=True, random_state=42).split(X, y)):
        Xtr, Xva, Xte = X.iloc[tr].copy(), X.iloc[va].copy(), X_test.copy()
        fold_id[va] = k
        if te_cols:
            enc_tr, enc_va, enc_te = target_encode(Xtr, y[tr], [Xva, Xte], te_cols, alpha=TE_ALPHA)
            for name in enc_tr:
                Xtr[name] = enc_tr[name]
                Xva[name] = enc_va[name]
                Xte[name] = enc_te[name]
        dtr = lgb.Dataset(Xtr, y[tr])
        dva = lgb.Dataset(Xva, y[va], reference=dtr)
        mdl = lgb.train(params, dtr, rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[va] = mdl.predict(Xva, num_iteration=mdl.best_iteration)
        pred += mdl.predict(Xte, num_iteration=mdl.best_iteration) / n_folds
        best.append(mdl.best_iteration)
        if log:
            print(f"  fold {k}: AUC {roc_auc_score(y[va], oof[va]):.5f}  ({mdl.best_iteration} rounds)", flush=True)
    return dict(oof=oof, pred=pred, fold=fold_id, auc=roc_auc_score(y, oof), rounds=best, model=mdl)


ADDITIVE = False   # True: interaction_constraints put every column in its own group (the ablation: costs 0.0001 against these params)
PARAMS = dict(objective="binary", metric="auc", learning_rate=0.1 if FAST else 0.02, num_leaves=32, max_depth=5, min_data_in_leaf=10,
              feature_fraction=0.3, bagging_fraction=0.8, bagging_freq=1, lambda_l2=2.0, lambda_l1=0.071, max_bin=1024, seed=SEED,
              num_threads=os.cpu_count(), verbose=-1)
if ADDITIVE:
    PARAMS["interaction_constraints"] = [[i] for i in range(X.shape[1] + len(TE_COLS))]
if FAST:
    idx = np.random.default_rng(SEED).choice(len(X), 60000, replace=False)
    X_run, y_run = X.iloc[idx].reset_index(drop=True), y[idx]
else:
    X_run, y_run = X, y
t0 = time.time()
res = run_cv(X_run, y_run, X_test, PARAMS, n_folds=3 if FAST else 5)
print(f"out-of-fold AUC {res['auc']:.5f}   ({(time.time() - t0) / 60:.1f} min)")

Gain share of the top features (last fold's model). The nearest-original key takes more gain than the exact value: it serves the 2.1% invented incomes as well.

In [ ]:
imp = pd.Series(res["model"].feature_importance("gain"), index=res["model"].feature_name()).sort_values(ascending=False)
imp = (imp / imp.sum()).round(4)
display(imp.head(12).to_frame("share of total gain"))

## 6. What 57,314 rows can resolve

The public board is scored on 20% of the test rows. Before reading a fourth-decimal difference between two files as a result, it helps to know what that many rows can tell apart. The cell trains a second real model (the additive reference of the ablation, at learning rate 0.1) and subsamples both out-of-fold vectors, without replacement, at the public size: the absolute uncertainty of one score, and the uncertainty of the *difference* between two files from the same pipeline — much smaller, because they are scored on the same rows and disagree only on a few of them; it grows with how much the two files disagree (megayak's §8 does this subsampling first and measured 0.00002 for two near-identical candidates; busyaprime fits the public SE two ways; Mamarin's ledger measures it from finished episodes). The three submissions behind this notebook: the v2 model (additive, prior 10, lr 0.02) 0.94564 → the additive reference at lr 0.02, 0.94584 → this model, 0.94603 out of fold; 0.94585 → 0.94595 → 0.94617 on the public board (+0.00020 / +0.00019 out of fold against +0.00010 / +0.00022 public). Pick final submissions on the out-of-fold number, which uses every training row.

In [ ]:
# What a public score computed on 57,314 rows (20% of test) can resolve. Absolute: bootstrap the AUC of this model's
# out-of-fold predictions on random 57,314-row subsets. Paired: the same for the DIFFERENCE between this model and a second
# real model - the additive one from the ablation, trained here at learning rate 0.1 - which is how two submissions from
# one pipeline differ: on specific rows, not by random noise.
from scipy.stats import spearmanr
P_ADD = dict(PARAMS, learning_rate=0.1, num_leaves=31, max_depth=-1, min_data_in_leaf=100, feature_fraction=0.8, lambda_l2=1.0,
             lambda_l1=0.0, max_bin=4095, interaction_constraints=[[i] for i in range(X.shape[1] + len(TE_COLS))])
t0 = time.time()
res_add = run_cv(X_run, y_run, X_test, P_ADD, n_folds=3 if FAST else 5, log=False)
oof, oof_add = res["oof"], res_add["oof"]
rho = spearmanr(oof, oof_add).statistic
print(f"second model (additive, learning rate 0.1): out-of-fold AUC {res_add['auc']:.5f}, Spearman with the final model {rho:.4f}  "
      f"({(time.time() - t0) / 60:.1f} min)")
n_pub = 57314
sub_auc, diff = [], []
for _ in range(300):
    idx = rng.choice(len(oof), n_pub, replace=False)
    a = roc_auc_score(y_run[idx], oof[idx])
    sub_auc.append(a)
    diff.append(a - roc_auc_score(y_run[idx], oof_add[idx]))
print(f"SD of an AUC measured on {n_pub:,} rows: {np.std(sub_auc):.5f}   (a public score is that uncertain in absolute terms)")
print(f"full out-of-fold gap between the two models: {res['auc'] - res_add['auc']:+.5f};  SD of that gap measured on {n_pub:,} rows: "
      f"{np.std(diff):.5f}   (the gap is {abs(res['auc'] - res_add['auc']) / np.std(diff):.1f} SDs: just resolvable on a public board, for this pair)")
print(f"for scale, a gap of 0.00005 between two such files would be {0.00005 / np.std(diff):.1f} SDs")

## 7. Outputs

`submission.csv` is the 5-fold average on test. `oof_generator_memory.csv` carries one prediction per training row and the fold it was scored in, on the same 5-fold split as above (`StratifiedKFold(5, shuffle=True, random_state=42)` over `train.csv` in file order — the split Mamarin's and dariushafshar's OOF libraries use; Naji and megayak publish on 10 folds, same seed). `train_original_map.csv` links the 76% of train rows whose income identifies a single original row to that row (`original_Buyer_ID`, `original_label`, `shared_recipe_cols`, `original_rows_with_this_income`, `income_is_original`, `nearest_original_income`) so the controls of §3–§4 can be redone or extended without this notebook.

## 8. What this does and does not say

* All of this describes the competition generator, not electric-vehicle buyers. On the original file the untrained recipe scores about 0.90 AUC and 0.938 on train, and none of the memory exists on the original — Chris Deotte's EDA and [Dariush Afshar's *0.938 phenomenon*](https://www.kaggle.com/code/dariushafshar/s6e9-0-938-phenomenon-0-946-generator) make that point with the right controls (Afshar splits 0.938 → 0.946 into a sharper rule, new dependencies on the other nine columns, and per-value artefacts). The raw thirteen columns at 255 bins reach 0.9418 without seeing any exact value; what sits above that is a description of how one synthetic-data generator remembers what it was trained on.
* The memory is measurable and it is not a lever. The training rows estimate the per-value part better than the original file can, and the per-row part is too thin to estimate at all. The practical content of this notebook is the encoding: light prior, several keys, a well-regularised booster.
* Commute (`Daily_Commute_km`) is in the encoding because its exact value also clears the bar in §5 (whole km, see the table); its own version of §2–§4 is left for someone else's afternoon.

In [ ]:
sub = sub[["id"]].merge(pd.DataFrame({"id": test.id, "Will_Buy_EV": res["pred"]}), on="id", how="left")
sub.to_csv("submission.csv", index=False)
if not FAST:
    pd.DataFrame({"id": train.id, "fold": res["fold"], "oof": res["oof"]}).to_csv("oof_generator_memory.csv", index=False)
    Ot = oo.reindex(train.Annual_Income_USD)
    n_orig = orig.Annual_Income_USD.value_counts()
    pd.DataFrame({"id": train.id, "original_Buyer_ID": Ot["Buyer_ID"].values,
                  "original_label": pd.array(Ot["y"].values, dtype="Int64"),
                  "shared_recipe_cols": pd.array(np.where(np.isnan(Ot["y"].values), np.nan, sum((train[c].values == Ot[c].values).astype("int8") for c in RECIPE)), dtype="Int64"),
                  "original_rows_with_this_income": train.Annual_Income_USD.map(n_orig).fillna(0).astype(int).values,
                  "income_is_original": train.Annual_Income_USD.isin(orig_values).values,
                  "nearest_original_income": nearest_original(train.Annual_Income_USD.values).astype(int)}).to_csv("train_original_map.csv", index=False)
print(sub.head(3))
print("written: submission.csv" + ("" if FAST else ", oof_generator_memory.csv (5 folds, seed 42, file order), train_original_map.csv"))